In [1]:
# ==========================================================
# Cell 1A : Import Libraries
# ==========================================================

import os
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")

print("="*70)
print("Libraries Imported Successfully")
print("="*70)

Libraries Imported Successfully


In [2]:
# ==========================================================
# Cell 1B : File Paths
# ==========================================================

DATA_PATH = r"C:\Users\JUNAID HABIB\OneDrive - Higher Education Commission\Desktop\FYP\EMI\ModelsForGraphene\Results\Data\Final_Dataset.xlsx"

MODEL_FOLDER = r"C:\Users\JUNAID HABIB\OneDrive - Higher Education Commission\Desktop\FYP\EMI\ModelsForGraphene\Results\Models"

MODEL_PATH = os.path.join(MODEL_FOLDER, "Final_GBR_Model.pkl")

FEATURE_PATH = os.path.join(MODEL_FOLDER, "Feature_Names.pkl")

print("Paths Loaded Successfully")

Paths Loaded Successfully


In [3]:
# ==========================================================
# Cell 1C : Load Dataset and Saved Model
# ==========================================================

df = pd.read_excel(DATA_PATH)

model = joblib.load(MODEL_PATH)

feature_names = joblib.load(FEATURE_PATH)

print("="*70)
print("Dataset Shape :", df.shape)
print("Model Loaded Successfully")
print("Total Features :", len(feature_names))
print("="*70)

Dataset Shape : (456, 10)
Model Loaded Successfully
Total Features : 10


In [4]:
# ==========================================================
# Cell 1D : Build Encoder from Final Dataset
# ==========================================================

categorical_columns = [

    "Matrix",
    "Filler1",
    "Filler2",
    "Followed by",
    "Structure"

]

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(df[categorical_columns])

print("="*70)
print("Fresh Encoder Created")
print("="*70)

for feature, cats in zip(categorical_columns, encoder.categories_):

    print(f"\n{feature}")
    print(list(cats))

Fresh Encoder Created

Matrix
['ABS', 'CS ', 'EP', 'EVA', 'P3HT', 'PA', 'PC', 'PEI', 'PES', 'PMMA', 'PP', 'PR', 'PS', 'PU', 'PVDF', 'SAE ', 'SEBS']

Filler1
['CB', 'CF', 'EG', 'EVA', 'Fe', 'Graphene', 'Graphite', 'MWCNT', 'RGO', 'SWCNT', 'non']

Filler2
['CB', 'CF', 'EG', 'Fe3O4', 'Fe3O4 ', 'MWCNT', 'non']

Followed by
['CAS', 'CM', 'IM']

Structure
['Common', 'Foam']


In [5]:
# ==========================================================
# Cell 1E : Final Consistency Check
# ==========================================================

print("="*70)

print("Dataset Features :", df.shape[1])

print("Model Features   :", len(feature_names))

print("Encoder Features :", len(categorical_columns))

print("="*70)

print("\nDataset Columns")

for col in df.columns:
    print("-", col)

print("\nModel Features")

for col in feature_names:
    print("-", col)

print("="*70)

Dataset Features : 10
Model Features   : 10
Encoder Features : 5

Dataset Columns
- Matrix
- Filler1
- % Loading1
- Filler2
- % Loading2
- Followed by
- Thickness (mm)
- Number of layer
- Structure
- Porosity

Model Features
- Matrix
- Filler1
- % Loading1
- Filler2
- % Loading2
- Followed by
- Thickness (mm)
- Number of layer
- Structure
- Porosity


In [6]:
# ==========================================================
# Cell 2A : Verify Encoder Mapping
# ==========================================================

for feature, cats in zip(categorical_columns, encoder.categories_):

    print("\n" + "="*70)
    print(feature)
    print("="*70)

    for code, category in enumerate(cats):

        print(f"{code:2d} --> {category}")


Matrix
 0 --> ABS
 1 --> CS 
 2 --> EP
 3 --> EVA
 4 --> P3HT
 5 --> PA
 6 --> PC
 7 --> PEI
 8 --> PES
 9 --> PMMA
10 --> PP
11 --> PR
12 --> PS
13 --> PU
14 --> PVDF
15 --> SAE 
16 --> SEBS

Filler1
 0 --> CB
 1 --> CF
 2 --> EG
 3 --> EVA
 4 --> Fe
 5 --> Graphene
 6 --> Graphite
 7 --> MWCNT
 8 --> RGO
 9 --> SWCNT
10 --> non

Filler2
 0 --> CB
 1 --> CF
 2 --> EG
 3 --> Fe3O4
 4 --> Fe3O4 
 5 --> MWCNT
 6 --> non

Followed by
 0 --> CAS
 1 --> CM
 2 --> IM

Structure
 0 --> Common
 1 --> Foam


In [7]:
# ==========================================================
# Cell 2D : Clean Categorical Variables
# ==========================================================

categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

print("="*70)
print("Categorical Features Cleaned Successfully")
print("="*70)

Categorical Features Cleaned Successfully


In [8]:
# ==========================================================
# Cell 3A : Prediction Function
# ==========================================================

def predict_se(candidate):
    """
    Predict EMI Shielding Effectiveness of a single candidate.
    """

    sample = pd.DataFrame([candidate])

    # Encode categorical variables
    sample[categorical_columns] = encoder.transform(sample[categorical_columns])

    # Ensure correct feature order
    sample = sample[feature_names]

    prediction = model.predict(sample)[0]

    return float(prediction)

In [9]:
# ==========================================================
# Cell 3B : Test Prediction
# ==========================================================

candidate = {

    "Matrix": "PVDF",

    "Filler1": "Graphene",

    "% Loading1": 8.0,

    "Filler2": "MWCNT",

    "% Loading2": 2.0,

    "Followed by": "CM",

    "Thickness (mm)": 1.5,

    "Number of layer": 2,

    "Structure": "Common",

    "Porosity": 5.0

}

prediction = predict_se(candidate)

print("="*70)
print(f"Predicted Shielding Effectiveness : {prediction:.2f} dB")
print("="*70)

Predicted Shielding Effectiveness : 18.76 dB


In [10]:
# ==========================================================
# Cell 3A : Evaluate Candidate
# ==========================================================

def evaluate_candidate(candidate):
    """
    Evaluate a candidate composite formulation.

    Parameters
    ----------
    candidate : dict

    Returns
    -------
    dict
        Predicted SE and useful optimization metrics.
    """

    sample = pd.DataFrame([candidate])

    # Clean strings
    for col in categorical_columns:
        sample[col] = sample[col].astype(str).str.strip()

    # Encode categorical variables
    sample[categorical_columns] = encoder.transform(sample[categorical_columns])

    # Arrange columns exactly as used during training
    sample = sample[feature_names]

    # Predict EMI Shielding
    predicted_se = float(model.predict(sample)[0])

    # Total filler loading
    total_loading = (
        candidate["% Loading1"] +
        candidate["% Loading2"]
    )

    return {

        "Predicted_SE": predicted_se,

        "Total_Loading": total_loading,

        "Candidate": candidate

    }

In [11]:
# ==========================================================
# Cell 3B : Test Evaluation Function
# ==========================================================

candidate = {

    "Matrix": "PVDF",

    "Filler1": "Graphene",

    "% Loading1": 8.0,

    "Filler2": "MWCNT",

    "% Loading2": 2.0,

    "Followed by": "CM",

    "Thickness (mm)": 1.5,

    "Number of layer": 2,

    "Structure": "Common",

    "Porosity": 5.0

}

result = evaluate_candidate(candidate)

print("="*70)

print("Evaluation Result")

print("="*70)

for k, v in result.items():

    if k != "Candidate":

        print(f"{k:20s}: {v}")

print("="*70)

Evaluation Result
Predicted_SE        : 18.75538687649261
Total_Loading       : 10.0


In [12]:
# ==========================================================
# Cell 4A : Import Optimization Libraries
# ==========================================================

import random
import numpy as np
import pandas as pd

from pymoo.core.problem import ElementwiseProblem

from pymoo.algorithms.moo.nsga2 import NSGA2

from pymoo.optimize import minimize

print("="*70)
print("Optimization Libraries Imported Successfully")
print("="*70)

Optimization Libraries Imported Successfully


In [13]:
# ==========================================================
# Cell 4AA : Build Search Space Dictionaries
# ==========================================================

# Categorical search space
categories = {
    "Matrix": sorted(df["Matrix"].dropna().unique().tolist()),
    "Filler1": sorted(df["Filler1"].dropna().unique().tolist()),
    "Filler2": sorted(df["Filler2"].dropna().unique().tolist()),
    "Followed by": sorted(df["Followed by"].dropna().unique().tolist()),
    "Structure": sorted(df["Structure"].dropna().unique().tolist())
}

# Continuous / Integer search space
bounds = {
    "% Loading1": (
        float(df["% Loading1"].min()),
        float(df["% Loading1"].max())
    ),
    "% Loading2": (
        float(df["% Loading2"].min()),
        float(df["% Loading2"].max())
    ),
    "Thickness (mm)": (
        float(df["Thickness (mm)"].min()),
        float(df["Thickness (mm)"].max())
    ),
    "Number of layer": (
        int(df["Number of layer"].min()),
        int(df["Number of layer"].max())
    ),
    "Porosity": (
        float(df["Porosity"].min()),
        float(df["Porosity"].max())
    )
}

print("="*70)
print("Search Space Created Successfully")
print("="*70)

Search Space Created Successfully


In [14]:
# ==========================================================
# Cell 4B : Optimization Search Space
# ==========================================================

search_space = {

    "Matrix": categories["Matrix"],

    "Filler1": categories["Filler1"],

    "Filler2": categories["Filler2"],

    "Followed by": categories["Followed by"],

    "Structure": categories["Structure"],

    "% Loading1": bounds["% Loading1"],

    "% Loading2": bounds["% Loading2"],

    "Number of layer": bounds["Number of layer"],

    "Porosity": bounds["Porosity"]

}

print("="*70)

print("Optimization Variables")

print("="*70)

for k,v in search_space.items():

    print(k,":",v)

Optimization Variables
Matrix : ['ABS', 'CS', 'EP', 'EVA', 'P3HT', 'PA', 'PC', 'PEI', 'PES', 'PMMA', 'PP', 'PR', 'PS', 'PU', 'PVDF', 'SAE', 'SEBS']
Filler1 : ['CB', 'CF', 'EG', 'EVA', 'Fe', 'Graphene', 'Graphite', 'MWCNT', 'RGO', 'SWCNT', 'non']
Filler2 : ['CB', 'CF', 'EG', 'Fe3O4', 'MWCNT', 'non']
Followed by : ['CAS', 'CM', 'IM']
Structure : ['Common', 'Foam']
% Loading1 : (0.0, 100.0)
% Loading2 : (0.0, 15.0)
Number of layer : (1, 10)
Porosity : (0.0, 98.8)


In [15]:
# ==========================================================
# Cell 4C : User Inputs
# ==========================================================

TARGET_SE = 50.0

FIXED_THICKNESS = 1.5

TOP_SOLUTIONS = 20

print("="*70)

print("Inverse Design Settings")

print("="*70)

print(f"Target EMI        : {TARGET_SE} dB")

print(f"Thickness         : {FIXED_THICKNESS} mm")

print(f"Solutions Needed  : {TOP_SOLUTIONS}")

Inverse Design Settings
Target EMI        : 50.0 dB
Thickness         : 1.5 mm
Solutions Needed  : 20


In [16]:
# ==========================================================
# Cell 4D : Extract Valid Material Systems
# ==========================================================

material_systems = (
    df[
        [
            "Matrix",
            "Filler1",
            "Filler2",
            "Followed by",
            "Structure"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("="*70)
print("Valid Material Systems")
print("="*70)

print(f"Unique combinations : {len(material_systems)}")

display(material_systems.head())

Valid Material Systems
Unique combinations : 86


,Matrix,Filler1,Filler2,Followed by,Structure
0,PVDF,non,non,CM,Common
1,PVDF,Fe,non,CM,Common
2,PR,non,non,CAS,Foam
3,PR,MWCNT,non,CAS,Foam
4,PR,MWCNT,Fe3O4,CAS,Foam


In [17]:
# ==========================================================
# Cell 4E : Define NSGA-II Optimization Problem
# ==========================================================

from pymoo.core.problem import ElementwiseProblem

class GrapheneInverseDesign(ElementwiseProblem):

    def __init__(self):

        super().__init__(

            n_var=5,

            n_obj=2,

            n_ieq_constr=2,

            xl=np.array([
                0,
                bounds["% Loading1"][0],
                bounds["% Loading2"][0],
                bounds["Number of layer"][0],
                bounds["Porosity"][0]
            ]),

            xu=np.array([
                len(material_systems)-1,
                bounds["% Loading1"][1],
                bounds["% Loading2"][1],
                bounds["Number of layer"][1],
                bounds["Porosity"][1]
            ])

        )

    def _evaluate(self, x, out, *args, **kwargs):

        idx = int(round(x[0]))

        loading1 = float(x[1])
        loading2 = float(x[2])

        layers = int(round(x[3]))
        porosity = float(x[4])

        system = material_systems.iloc[idx]

        candidate = {

            "Matrix": system["Matrix"],
            "Filler1": system["Filler1"],
            "% Loading1": loading1,
            "Filler2": system["Filler2"],
            "% Loading2": loading2,
            "Followed by": system["Followed by"],
            "Thickness (mm)": FIXED_THICKNESS,
            "Number of layer": layers,
            "Structure": system["Structure"],
            "Porosity": porosity

        }

        result = evaluate_candidate(candidate)

        predicted = result["Predicted_SE"]

        total_loading = result["Total_Loading"]

        # Objective 1
        f1 = abs(predicted - TARGET_SE)

        # Objective 2
        f2 = total_loading

        # Constraints
        g1 = loading1 + loading2 - 100

        if system["Filler2"] == "non":
            g2 = loading2
        else:
            g2 = -1

        out["F"] = [f1, f2]
        out["G"] = [g1, g2]

In [18]:
# ==========================================================
# Cell 4E : Define NSGA-II Optimization Problem
# ==========================================================

from pymoo.core.problem import ElementwiseProblem

class GrapheneInverseDesign(ElementwiseProblem):

    def __init__(self):

        super().__init__(

            n_var=5,

            n_obj=2,

            n_ieq_constr=2,

            xl=np.array([
                0,
                bounds["% Loading1"][0],
                bounds["% Loading2"][0],
                bounds["Number of layer"][0],
                bounds["Porosity"][0]
            ]),

            xu=np.array([
                len(material_systems)-1,
                bounds["% Loading1"][1],
                bounds["% Loading2"][1],
                bounds["Number of layer"][1],
                bounds["Porosity"][1]
            ])

        )

    def _evaluate(self, x, out, *args, **kwargs):

        idx = int(round(x[0]))

        loading1 = float(x[1])
        loading2 = float(x[2])

        layers = int(round(x[3]))
        porosity = float(x[4])

        system = material_systems.iloc[idx]

        candidate = {

            "Matrix": system["Matrix"],
            "Filler1": system["Filler1"],
            "% Loading1": loading1,
            "Filler2": system["Filler2"],
            "% Loading2": loading2,
            "Followed by": system["Followed by"],
            "Thickness (mm)": FIXED_THICKNESS,
            "Number of layer": layers,
            "Structure": system["Structure"],
            "Porosity": porosity

        }

        result = evaluate_candidate(candidate)

        predicted = result["Predicted_SE"]

        total_loading = result["Total_Loading"]

        # Objective 1
        f1 = abs(predicted - TARGET_SE)

        # Objective 2
        f2 = total_loading

        # Constraints
        g1 = loading1 + loading2 - 100

        if system["Filler2"] == "non":
            g2 = loading2
        else:
            g2 = -1

        out["F"] = [f1, f2]
        out["G"] = [g1, g2]

In [19]:
# ==========================================================
# Cell 4G : Configure NSGA-II
# ==========================================================

from pymoo.algorithms.moo.nsga2 import NSGA2

algorithm = NSGA2(

    pop_size=120,

    eliminate_duplicates=True

)

problem = GrapheneInverseDesign()

print("="*70)
print("NSGA-II Configured Successfully")
print("="*70)

print("Population Size :", algorithm.pop_size)
print("Material Systems:", len(material_systems))
print("Objectives      : 2")
print("="*70)

NSGA-II Configured Successfully
Population Size : 120
Material Systems: 86
Objectives      : 2


In [20]:
# ==========================================================
# Cell 4H : Run NSGA-II Optimization
# ==========================================================

from pymoo.optimize import minimize
from pymoo.termination import get_termination

termination = get_termination(
    "n_gen",
    150          # Number of generations
)

result = minimize(

    problem,

    algorithm,

    termination,

    seed=42,

    save_history=True,

    verbose=True

)

print("="*70)
print("Optimization Completed Successfully")
print("="*70)

print("Pareto Solutions Found :", len(result.F))

print("="*70)

n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |      120 |      4 |  0.000000E+00 |  7.2904140373 |             - |             -
     2 |      240 |      5 |  0.000000E+00 |  2.0224922815 |  0.0801926388 |             f
     3 |      360 |      8 |  0.000000E+00 |  0.6037215187 |  0.0498467784 |         ideal
     4 |      480 |      8 |  0.000000E+00 |  0.0397222426 |  0.1358650921 |         ideal
     5 |      600 |     14 |  0.000000E+00 |  0.000000E+00 |  0.0173296970 |             f
     6 |      720 |     18 |  0.000000E+00 |  0.000000E+00 |  0.0327005615 |         ideal
     7 |      840 |     25 |  0.000000E+00 |  0.000000E+00 |  0.0247514815 |             f
     8 |      960 |     19 |  0.000000E+00 |  0.000000E+00 |  0.0074141941 |         ideal
     9 |     1080 |     14 |  0.000000E+00 |  0.000000E+00 |  0.0137437228 |         ideal
    10 |     1200 |     20 |  0.000000E+00 |  0.000000E+00 |  0.0225957041 |         nadir

    89 |    10680 |    119 |  0.000000E+00 |  0.000000E+00 |  0.0014227296 |             f
    90 |    10800 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0014857650 |             f
    91 |    10920 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0013732112 |             f
    92 |    11040 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0014732846 |             f
    93 |    11160 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0018995376 |             f
    94 |    11280 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0019569571 |             f
    95 |    11400 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0019947423 |             f
    96 |    11520 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0020233928 |             f
    97 |    11640 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0020195824 |             f
    98 |    11760 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0020177131 |             f
    99 |    11880 |    120 |  0.000000E+00 |  0.000000E+00 |  0.0021930178 |             f

In [21]:
# ==========================================================
# Cell 4I : Decode Pareto Solutions
# ==========================================================

pareto_results = []

for x, f in zip(result.X, result.F):

    idx = int(round(x[0]))

    system = material_systems.iloc[idx]

    loading1 = float(x[1])

    loading2 = float(x[2])

    if system["Filler1"] == "non":
        loading1 = 0.0

    if system["Filler2"] == "non":
        loading2 = 0.0

    candidate = {

        "Matrix": system["Matrix"],
        "Filler1": system["Filler1"],
        "% Loading1": loading1,
        "Filler2": system["Filler2"],
        "% Loading2": loading2,
        "Followed by": system["Followed by"],
        "Thickness (mm)": FIXED_THICKNESS,
        "Number of layer": int(round(x[3])),
        "Structure": system["Structure"],
        "Porosity": float(x[4])

    }

    pred = evaluate_candidate(candidate)

    pareto_results.append({

        **candidate,

        "Predicted_SE": pred["Predicted_SE"],

        "SE_Error": abs(pred["Predicted_SE"] - TARGET_SE),

        "Total_Loading": pred["Total_Loading"]

    })

pareto_df = pd.DataFrame(pareto_results)

pareto_df = pareto_df.sort_values(

    by=["SE_Error", "Total_Loading"]

).reset_index(drop=True)

print("="*70)
print("Top Recommended Composite Designs")
print("="*70)

display(pareto_df.head(TOP_SOLUTIONS))

Top Recommended Composite Designs


,Matrix,Filler1,% Loading1,Filler2,% Loading2,Followed by,Thickness (mm),Number of layer,Structure,Porosity,Predicted_SE,SE_Error,Total_Loading
0,PR,MWCNT,73.284972,Fe3O4,0.353484,CAS,1.5,5,Foam,4.984635,49.992824,0.007176,73.638457
1,PS,Graphite,70.651192,MWCNT,0.160890,CM,1.5,6,Common,32.901272,49.987740,0.012260,70.812082
2,PS,Graphite,70.293464,MWCNT,0.057759,CM,1.5,6,Common,29.615068,50.021039,0.021039,70.351222
3,PS,Graphite,70.293464,MWCNT,0.057759,CM,1.5,6,Common,29.615068,50.021039,0.021039,70.351222
4,PS,Graphite,70.251036,MWCNT,0.001056,CM,1.5,6,Common,33.249791,49.916284,0.083716,70.252092
5,PS,Graphite,68.310332,MWCNT,0.755022,CM,1.5,5,Common,0.388429,48.386981,1.613019,69.065354
6,PS,Graphite,67.710578,MWCNT,0.755134,CM,1.5,5,Common,0.193164,48.359168,1.640832,68.465712
7,PS,Graphite,18.256709,MWCNT,6.033738,CM,1.5,5,Common,28.681875,48.301110,1.698890,24.290447
8,PS,Graphite,18.256709,MWCNT,5.031070,CM,1.5,5,Common,29.646424,48.278569,1.721431,23.287779
9,PS,Graphite,18.081884,MWCNT,5.087371,CM,1.5,5,Common,29.163124,48.248097,1.751903,23.169255


In [22]:
# ==========================================================
# Cell 5A : Remove Duplicate Solutions
# ==========================================================

# Create a copy
clean_df = pareto_df.copy()

# Round numerical variables
clean_df["% Loading1"] = clean_df["% Loading1"].round(2)
clean_df["% Loading2"] = clean_df["% Loading2"].round(2)
clean_df["Porosity"] = clean_df["Porosity"].round(2)
clean_df["Predicted_SE"] = clean_df["Predicted_SE"].round(2)

# Remove duplicate formulations
clean_df = clean_df.drop_duplicates(

    subset=[

        "Matrix",
        "Filler1",
        "% Loading1",
        "Filler2",
        "% Loading2",
        "Followed by",
        "Thickness (mm)",
        "Number of layer",
        "Structure",
        "Porosity"

    ],

    keep="first"

)

clean_df = clean_df.reset_index(drop=True)

print("="*70)
print("Duplicate Removal Summary")
print("="*70)

print(f"Original Solutions : {len(pareto_df)}")
print(f"Unique Solutions   : {len(clean_df)}")
print(f"Duplicates Removed : {len(pareto_df)-len(clean_df)}")

print("="*70)

display(clean_df.head(20))

Duplicate Removal Summary
Original Solutions : 120
Unique Solutions   : 91
Duplicates Removed : 29


,Matrix,Filler1,% Loading1,Filler2,% Loading2,Followed by,Thickness (mm),Number of layer,Structure,Porosity,Predicted_SE,SE_Error,Total_Loading
0,PR,MWCNT,73.28,Fe3O4,0.35,CAS,1.5,5,Foam,4.98,49.99,0.007176,73.638457
1,PS,Graphite,70.65,MWCNT,0.16,CM,1.5,6,Common,32.90,49.99,0.012260,70.812082
2,PS,Graphite,70.29,MWCNT,0.06,CM,1.5,6,Common,29.62,50.02,0.021039,70.351222
3,PS,Graphite,70.25,MWCNT,0.00,CM,1.5,6,Common,33.25,49.92,0.083716,70.252092
4,PS,Graphite,68.31,MWCNT,0.76,CM,1.5,5,Common,0.39,48.39,1.613019,69.065354
5,PS,Graphite,67.71,MWCNT,0.76,CM,1.5,5,Common,0.19,48.36,1.640832,68.465712
6,PS,Graphite,18.26,MWCNT,6.03,CM,1.5,5,Common,28.68,48.30,1.698890,24.290447
7,PS,Graphite,18.26,MWCNT,5.03,CM,1.5,5,Common,29.65,48.28,1.721431,23.287779
8,PS,Graphite,18.08,MWCNT,5.09,CM,1.5,5,Common,29.16,48.25,1.751903,23.169255
9,PS,Graphite,15.07,MWCNT,6.04,CM,1.5,5,Common,29.60,48.23,1.773382,21.111195


In [23]:
# ==========================================================
# Cell 5B : Engineering Validation
# ==========================================================

validated = pareto_df.copy()

# ----------------------------------------------------------
# Dataset Limits
# ----------------------------------------------------------

loading1_min, loading1_max = df["% Loading1"].min(), df["% Loading1"].max()
loading2_min, loading2_max = df["% Loading2"].min(), df["% Loading2"].max()

porosity_min, porosity_max = df["Porosity"].min(), df["Porosity"].max()

layer_min, layer_max = (
    df["Number of layer"].min(),
    df["Number of layer"].max()
)

# ----------------------------------------------------------
# Total Loading
# ----------------------------------------------------------

validated["Total_Loading"] = (
    validated["% Loading1"] +
    validated["% Loading2"]
)

# ----------------------------------------------------------
# Engineering Constraints
# ----------------------------------------------------------

validated = validated[

    (validated["% Loading1"] >= loading1_min) &
    (validated["% Loading1"] <= loading1_max) &

    (validated["% Loading2"] >= loading2_min) &
    (validated["% Loading2"] <= loading2_max) &

    (validated["Porosity"] >= porosity_min) &
    (validated["Porosity"] <= porosity_max) &

    (validated["Number of layer"] >= layer_min) &
    (validated["Number of layer"] <= layer_max)

].copy()

# ----------------------------------------------------------
# Remove Invalid Predictions
# ----------------------------------------------------------

validated = validated.dropna(subset=["Predicted_SE"])

validated = validated[
    validated["Predicted_SE"] > 0
]

validated = validated[
    validated["SE_Error"] >= 0
]

validated = validated.reset_index(drop=True)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print("="*70)
print("Engineering Validation Completed")
print("="*70)

print("Solutions Before :", len(pareto_df))
print("Solutions After  :", len(validated))

print("="*70)

display(validated.head())

Engineering Validation Completed
Solutions Before : 120
Solutions After  : 120


,Matrix,Filler1,% Loading1,Filler2,% Loading2,Followed by,Thickness (mm),Number of layer,Structure,Porosity,Predicted_SE,SE_Error,Total_Loading
0,PR,MWCNT,73.284972,Fe3O4,0.353484,CAS,1.5,5,Foam,4.984635,49.992824,0.007176,73.638457
1,PS,Graphite,70.651192,MWCNT,0.160890,CM,1.5,6,Common,32.901272,49.987740,0.012260,70.812082
2,PS,Graphite,70.293464,MWCNT,0.057759,CM,1.5,6,Common,29.615068,50.021039,0.021039,70.351222
3,PS,Graphite,70.293464,MWCNT,0.057759,CM,1.5,6,Common,29.615068,50.021039,0.021039,70.351222
4,PS,Graphite,70.251036,MWCNT,0.001056,CM,1.5,6,Common,33.249791,49.916284,0.083716,70.252092


In [24]:
# ==========================================================
# Cell 5C : Intelligent Engineering Ranking
# ==========================================================

ranking = validated.copy()

# ----------------------------------------------------------
# Normalize Metrics (0 → 1)
# Lower values are better
# ----------------------------------------------------------

ranking["Norm_Error"] = (
    ranking["SE_Error"] -
    ranking["SE_Error"].min()
) / (
    ranking["SE_Error"].max() -
    ranking["SE_Error"].min() + 1e-9
)

ranking["Norm_Loading"] = (
    ranking["Total_Loading"] -
    ranking["Total_Loading"].min()
) / (
    ranking["Total_Loading"].max() -
    ranking["Total_Loading"].min() + 1e-9
)

ranking["Norm_Porosity"] = (
    ranking["Porosity"] -
    ranking["Porosity"].min()
) / (
    ranking["Porosity"].max() -
    ranking["Porosity"].min() + 1e-9
)

ranking["Norm_Layers"] = (
    ranking["Number of layer"] -
    ranking["Number of layer"].min()
) / (
    ranking["Number of layer"].max() -
    ranking["Number of layer"].min() + 1e-9
)

# ----------------------------------------------------------
# Engineering Score
# Lower score = Better material
# ----------------------------------------------------------

ranking["Engineering_Score"] = (

      0.50 * ranking["Norm_Error"]
    + 0.25 * ranking["Norm_Loading"]
    + 0.15 * ranking["Norm_Porosity"]
    + 0.10 * ranking["Norm_Layers"]

)

# ----------------------------------------------------------
# Final Ranking
# ----------------------------------------------------------

ranking = ranking.sort_values(
    by="Engineering_Score"
).reset_index(drop=True)

ranking.insert(0, "Rank", range(1, len(ranking)+1))

print("="*70)
print("Engineering Ranking Completed")
print("="*70)

display(
    ranking.head(10)
)

Engineering Ranking Completed


,Rank,Matrix,Filler1,% Loading1,Filler2,% Loading2,Followed by,Thickness (mm),Number of layer,Structure,Porosity,Predicted_SE,SE_Error,Total_Loading,Norm_Error,Norm_Loading,Norm_Porosity,Norm_Layers,Engineering_Score
0,1,PS,Graphite,15.086061,MWCNT,0.752047,CM,1.5,5,Common,1.922456,46.741715,3.258285,15.838107,0.090719,0.215079,0.044342,0.0,0.105781
1,2,PS,Graphite,12.627951,MWCNT,0.755134,CM,1.5,5,Common,0.388429,45.577810,4.422190,13.383085,0.123197,0.181740,0.005007,0.0,0.107785
2,3,PS,Graphite,12.539958,MWCNT,0.755134,CM,1.5,5,Common,0.193164,45.387653,4.612347,13.295092,0.128503,0.180546,0.000000,0.0,0.109388
3,4,PS,Graphite,12.627951,MWCNT,0.755134,CM,1.5,5,Common,1.473079,45.577810,4.422190,13.383085,0.123197,0.181740,0.032819,0.0,0.111957
4,5,PS,Graphite,14.589334,MWCNT,0.755609,CM,1.5,5,Common,7.524873,46.636765,3.363235,15.344942,0.093648,0.208382,0.187998,0.0,0.127119
5,6,PS,Graphite,14.589334,MWCNT,0.755609,CM,1.5,5,Common,7.524873,46.636765,3.363235,15.344942,0.093648,0.208382,0.187998,0.0,0.127119
6,7,PS,Graphite,13.527878,MWCNT,0.755259,CM,1.5,5,Common,5.969771,45.814275,4.185725,14.283137,0.116599,0.193963,0.148123,0.0,0.129009
7,8,PR,MWCNT,1.791261,Fe3O4,6.001364,CAS,1.5,5,Foam,29.785693,45.167954,4.832046,7.792625,0.134634,0.105823,0.758806,0.0,0.207594
8,9,PR,MWCNT,1.791261,Fe3O4,6.001364,CAS,1.5,5,Foam,29.785693,45.167954,4.832046,7.792625,0.134634,0.105823,0.758806,0.0,0.207594
9,10,PS,Graphite,14.503931,MWCNT,6.040322,CM,1.5,5,Common,29.610822,48.104647,1.895353,20.544253,0.052688,0.278988,0.754322,0.0,0.209239


In [25]:
# ==========================================================
# Cell 5D : Generate Final Recommendation Table
# ==========================================================

TOP_K = 20

recommendations = ranking.head(TOP_K).copy()

# ----------------------------------------------------------
# Keep only useful columns
# ----------------------------------------------------------

recommendations = recommendations[[
    "Rank",
    "Matrix",
    "Filler1",
    "% Loading1",
    "Filler2",
    "% Loading2",
    "Followed by",
    "Thickness (mm)",
    "Number of layer",
    "Structure",
    "Porosity",
    "Predicted_SE",
    "SE_Error",
    "Total_Loading",
    "Engineering_Score"
]]

# ----------------------------------------------------------
# Round numerical columns
# ----------------------------------------------------------

numeric_cols = [
    "% Loading1",
    "% Loading2",
    "Thickness (mm)",
    "Porosity",
    "Predicted_SE",
    "SE_Error",
    "Total_Loading",
    "Engineering_Score"
]

recommendations[numeric_cols] = recommendations[numeric_cols].round(3)

# ----------------------------------------------------------
# Display
# ----------------------------------------------------------

print("="*80)
print(f"Top {TOP_K} Recommended EMI Material Systems")
print("="*80)

display(recommendations)

print("="*80)

Top 20 Recommended EMI Material Systems


,Rank,Matrix,Filler1,% Loading1,Filler2,% Loading2,Followed by,Thickness (mm),Number of layer,Structure,Porosity,Predicted_SE,SE_Error,Total_Loading,Engineering_Score
0,1,PS,Graphite,15.086,MWCNT,0.752,CM,1.5,5,Common,1.922,46.742,3.258,15.838,0.106
1,2,PS,Graphite,12.628,MWCNT,0.755,CM,1.5,5,Common,0.388,45.578,4.422,13.383,0.108
2,3,PS,Graphite,12.540,MWCNT,0.755,CM,1.5,5,Common,0.193,45.388,4.612,13.295,0.109
3,4,PS,Graphite,12.628,MWCNT,0.755,CM,1.5,5,Common,1.473,45.578,4.422,13.383,0.112
4,5,PS,Graphite,14.589,MWCNT,0.756,CM,1.5,5,Common,7.525,46.637,3.363,15.345,0.127
5,6,PS,Graphite,14.589,MWCNT,0.756,CM,1.5,5,Common,7.525,46.637,3.363,15.345,0.127
6,7,PS,Graphite,13.528,MWCNT,0.755,CM,1.5,5,Common,5.970,45.814,4.186,14.283,0.129
7,8,PR,MWCNT,1.791,Fe3O4,6.001,CAS,1.5,5,Foam,29.786,45.168,4.832,7.793,0.208
8,9,PR,MWCNT,1.791,Fe3O4,6.001,CAS,1.5,5,Foam,29.786,45.168,4.832,7.793,0.208
9,10,PS,Graphite,14.504,MWCNT,6.040,CM,1.5,5,Common,29.611,48.105,1.895,20.544,0.209


In [26]:
# ==========================================================
# Cell 5E : Export Recommendations
# ==========================================================

import os

os.makedirs("Results/Inverse_Design", exist_ok=True)

output_file = "Results/Inverse_Design/Recommended_Materials.xlsx"

recommendations.to_excel(
    output_file,
    index=False
)

print("="*70)
print("Recommendations Saved Successfully")
print("="*70)
print(output_file)
print("="*70)

Recommendations Saved Successfully
Results/Inverse_Design/Recommended_Materials.xlsx


In [27]:
def inverse_design(
        target_se,
        thickness,
        n_solutions=20,
        pop_size=120,
        generations=150
):
    """
    Complete inverse design workflow.

    Parameters
    ----------
    target_se : float
        Desired EMI shielding (dB)

    thickness : float
        Desired thickness (mm)

    Returns
    -------
    recommendations : DataFrame
    """

    # ------------------------------------
    # Update user inputs
    # ------------------------------------

    USER_TARGET_SE = target_se
    USER_THICKNESS = thickness

    # ------------------------------------
    # Run NSGA-II
    # ------------------------------------

    algorithm = NSGA2(
        pop_size=pop_size
    )

    result = minimize(
        problem,
        algorithm,
        termination=("n_gen", generations),
        seed=42,
        verbose=False
    )

    # ------------------------------------
    # Decode Solutions
    # ------------------------------------

    pareto_df = decode_solutions(result.X)

    # ------------------------------------
    # Validation
    # ------------------------------------

    validated = engineering_validation(pareto_df)

    # ------------------------------------
    # Ranking
    # ------------------------------------

    ranking = engineering_ranking(validated)

    recommendations = build_recommendation_table(
        ranking,
        top_k=n_solutions
    )

    return recommendations

In [32]:
# ==========================================================
# Cell 6A : Complete Inverse Design Function
# ==========================================================

def inverse_design(target_se,
                   thickness,
                   n_solutions=20,
                   pop_size=120,
                   n_generations=150):

    """
    Performs complete inverse design.

    Parameters
    ----------
    target_se : float
    thickness : float

    Returns
    -------
    recommendations : pandas.DataFrame
    """

    global USER_TARGET_SE
    global USER_THICKNESS

    USER_TARGET_SE = float(target_se)
    USER_THICKNESS = float(thickness)

    # ---------------------------------------------
    # Build optimization problem
    # ---------------------------------------------

    problem = GrapheneInverseDesign()

    algorithm = NSGA2(
        pop_size=pop_size
    )

    result = minimize(
        problem,
        algorithm,
        termination=("n_gen", n_generations),
        seed=42,
        verbose=False
    )

    # ---------------------------------------------
    # Decode solutions
    # ---------------------------------------------

    pareto_df = decode_solutions(result.X)

    # ---------------------------------------------
    # Remove duplicates
    # ---------------------------------------------

    pareto_df = remove_duplicate_solutions(pareto_df)

    # ---------------------------------------------
    # Engineering validation
    # ---------------------------------------------

    validated = engineering_validation(pareto_df)

    # ---------------------------------------------
    # Engineering ranking
    # ---------------------------------------------

    ranking = engineering_ranking(validated)

    # ---------------------------------------------
    # Final recommendations
    # ---------------------------------------------

    recommendations = build_recommendation_table(
        ranking,
        top_k=n_solutions
    )

    return recommendations

In [33]:
table7 = recommendations[[
    "Rank",
    "Matrix",
    "Filler1",
    "% Loading1",
    "Filler2",
    "% Loading2",
    "Thickness (mm)",
    "Predicted_SE"
]].copy()

table7 = table7.rename(columns={
    "Filler1": "Primary Filler",
    "% Loading1": "Loading₁ (%)",
    "Filler2": "Secondary Filler",
    "% Loading2": "Loading₂ (%)",
    "Predicted_SE": "Predicted SE (dB)"
})

display(table7)

,Rank,Matrix,Primary Filler,Loading₁ (%),Secondary Filler,Loading₂ (%),Thickness (mm),Predicted SE (dB)
0,1,PS,Graphite,15.086,MWCNT,0.752,1.5,46.742
1,2,PS,Graphite,12.628,MWCNT,0.755,1.5,45.578
2,3,PS,Graphite,12.540,MWCNT,0.755,1.5,45.388
3,4,PS,Graphite,12.628,MWCNT,0.755,1.5,45.578
4,5,PS,Graphite,14.589,MWCNT,0.756,1.5,46.637
5,6,PS,Graphite,14.589,MWCNT,0.756,1.5,46.637
6,7,PS,Graphite,13.528,MWCNT,0.755,1.5,45.814
7,8,PR,MWCNT,1.791,Fe3O4,6.001,1.5,45.168
8,9,PR,MWCNT,1.791,Fe3O4,6.001,1.5,45.168
9,10,PS,Graphite,14.504,MWCNT,6.040,1.5,48.105


In [35]:
#table7.to_excel(
#    "Results/Inverse_Design/Table7_Recommended_Formulations.xlsx",
#    index=False
# )

In [36]:
# ==========================================================
# Cell 6B : Generate Publication Table (Table 7)
# ==========================================================

targets = [30, 50, 70]

table7 = []

for target in targets:

    rec = inverse_design(
        target_se=target,
        thickness=1.5,
        n_solutions=1
    )

    rec["Target SE (dB)"] = target

    table7.append(rec)

table7 = pd.concat(table7, ignore_index=True)

table7 = table7[[
    "Target SE (dB)",
    "Matrix",
    "Filler1",
    "% Loading1",
    "Filler2",
    "% Loading2",
    "Thickness (mm)",
    "Predicted_SE"
]]

table7.columns = [
    "Target SE (dB)",
    "Matrix",
    "Primary Filler",
    "Loading₁ (%)",
    "Secondary Filler",
    "Loading₂ (%)",
    "Thickness (mm)",
    "Predicted SE (dB)"
]

display(table7)

NameError: name 'decode_solutions' is not defined

In [37]:
# ==========================================================
# Cell 5F : Generate Publication Table (Table 7)
# ==========================================================

# Number of recommendations to show in the paper
TOP_N = 10

table7 = recommendations.head(TOP_N).copy()

# Keep only publication-relevant columns
table7 = table7[[
    "Rank",
    "Matrix",
    "Filler1",
    "% Loading1",
    "Filler2",
    "% Loading2",
    "Thickness (mm)",
    "Predicted_SE"
]]

# Rename columns
table7.columns = [
    "Rank",
    "Matrix",
    "Primary Filler",
    "Loading1 (%)",
    "Secondary Filler",
    "Loading2 (%)",
    "Thickness (mm)",
    "Predicted SE (dB)"
]

# Round numerical columns
for col in ["Loading1 (%)", "Loading2 (%)",
            "Thickness (mm)", "Predicted SE (dB)"]:
    table7[col] = table7[col].round(2)

print("="*80)
print("Table 7: Recommended Material Formulations")
print("="*80)

display(table7)

Table 7: Recommended Material Formulations


,Rank,Matrix,Primary Filler,Loading1 (%),Secondary Filler,Loading2 (%),Thickness (mm),Predicted SE (dB)
0,1,PS,Graphite,15.09,MWCNT,0.75,1.5,46.74
1,2,PS,Graphite,12.63,MWCNT,0.76,1.5,45.58
2,3,PS,Graphite,12.54,MWCNT,0.76,1.5,45.39
3,4,PS,Graphite,12.63,MWCNT,0.76,1.5,45.58
4,5,PS,Graphite,14.59,MWCNT,0.76,1.5,46.64
5,6,PS,Graphite,14.59,MWCNT,0.76,1.5,46.64
6,7,PS,Graphite,13.53,MWCNT,0.76,1.5,45.81
7,8,PR,MWCNT,1.79,Fe3O4,6.00,1.5,45.17
8,9,PR,MWCNT,1.79,Fe3O4,6.00,1.5,45.17
9,10,PS,Graphite,14.50,MWCNT,6.04,1.5,48.10
